# Part 2: How Real VLAs Represent Actions

## Notebook 3 — ACT (Action Chunking Transformer)

ACT (Zhao et al., RSS 2023) predicts **continuous action chunks** using a Conditional Variational Autoencoder (CVAE). Actions are raw continuous vectors — there is no tokenization step.

We load ACT from leRobot v0.6.0 and inspect its action handling.


### 1. AE, VAE, CVAE, and KL Divergence

ACT uses a Conditional Variational Autoencoder (CVAE) as its action head. Before we load ACT, let's build the intuition from the ground up:

- **AE (Autoencoder)**: compress input through a bottleneck, then reconstruct. Loss = MSE. The latent space has no structure.
- **VAE (Variational Autoencoder)**: the encoder outputs a distribution (μ, σ). We sample z = μ + σ·ε and add a KL divergence term to push the distribution toward N(0,1). This regularizes the latent space so nearby latents correspond to similar outputs.
- **CVAE (Conditional VAE)**: conditions both encoder and decoder on an observation. ACT feeds in camera images and joint states as the condition, so the latent z captures the action distribution for a specific situation.


In [1]:
import torch
import torch.nn as nn

# ── Toy data: circle of 2D points ──
torch.manual_seed(42)
n = 500
theta = torch.rand(n) * 2 * torch.pi
radius = 1.0 + 0.1 * torch.randn(n)
data = torch.stack([radius * torch.cos(theta), radius * torch.sin(theta)], dim=1)

# ── Autoencoder (AE) ──
class AE(nn.Module):
    def __init__(self, input_dim=2, latent_dim=1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16), nn.ReLU(),
            nn.Linear(16, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16), nn.ReLU(),
            nn.Linear(16, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = AE(latent_dim=1)
opt = torch.optim.Adam(ae.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for step in range(500):
    opt.zero_grad()
    loss = loss_fn(ae(data), data)
    loss.backward()
    opt.step()

with torch.no_grad():
    z_ae = ae.encoder(data)
    recon_ae = ae(data)

print(f"AE reconstruction MSE: {loss_fn(recon_ae, data):.4f}")
print(f"Latent z range: [{z_ae.min():.2f}, {z_ae.max():.2f}]")
print(f"Latent z mean/std: {z_ae.mean():.3f} / {z_ae.std():.3f}")
print(f"No regularization — the latent space is unstructured.")


AE reconstruction MSE: 0.0343
Latent z range: [-0.91, 5.54]
Latent z mean/std: 0.586 / 1.388
No regularization — the latent space is unstructured.


In [2]:
# ── Variational Autoencoder (VAE) ──
# Same architecture, but encoder outputs mu and log_var.
# We sample z = mu + sigma * epsilon (reparameterization trick).
# Loss = reconstruction_MSE + beta * KL_divergence

class VAE(nn.Module):
    def __init__(self, input_dim=2, latent_dim=1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16), nn.ReLU(),
        )
        self.mu_head = nn.Linear(16, latent_dim)
        self.logvar_head = nn.Linear(16, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16), nn.ReLU(),
            nn.Linear(16, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.mu_head(h), self.logvar_head(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

vae = VAE(latent_dim=1)
opt = torch.optim.Adam(vae.parameters(), lr=0.01)

for step in range(1000):
    opt.zero_grad()
    recon, mu, logvar = vae(data)
    recon_loss = loss_fn(recon, data)
    # KL divergence: D_KL( N(mu,sigma) || N(0,1) )
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / n
    loss = recon_loss + 0.05 * kl_loss
    loss.backward()
    opt.step()

with torch.no_grad():
    recon_vae, mu, logvar = vae(data)
    kl_value = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / n

print(f"VAE reconstruction MSE: {loss_fn(recon_vae, data):.4f}")
print(f"KL divergence: {kl_value:.4f}")
print(f"VAE latent: mu={mu.mean():.3f} ± {torch.exp(0.5 * logvar).mean():.3f}")
print(f"AE latent:  mu={z_ae.mean():.3f} ± {z_ae.std():.3f}")
print(f"\nKL divergence forces the latent toward N(0,1).")
print(f"The VAE latent is smoother and better regularized than the AE latent.")
print(f"\nCVAE adds conditioning: both encoder and decoder receive")
print(f"the observation as input. ACT conditions on images + joint states")
print(f"so the latent z captures actions appropriate for what the robot sees.")


VAE reconstruction MSE: 0.0771
KL divergence: 1.8535
VAE latent: mu=0.007 ± 0.179
AE latent:  mu=0.586 ± 1.388

KL divergence forces the latent toward N(0,1).
The VAE latent is smoother and better regularized than the AE latent.

CVAE adds conditioning: both encoder and decoder receive
the observation as input. ACT conditions on images + joint states
so the latent z captures actions appropriate for what the robot sees.


### 2. Load ACT configuration

ACT is a policy that predicts chunks of actions directly via continuous regression.


In [1]:
from lerobot.policies.act.configuration_act import ACTConfig

cfg = ACTConfig()
print(f"Policy type: ACT (Action Chunking Transformer)")
print(f"Chunk size:        {cfg.chunk_size}")  # 100
print(f"Action steps:      {cfg.n_action_steps}")  # 100
print(f"Input features:    {cfg.input_features}")
print(f"Output features:   {cfg.output_features}")


Policy type: ACT (Action Chunking Transformer)
Chunk size:        100
Action steps:      100
Input features:    {}
Output features:   {}


### 3. Action representation: pure continuous

ACT outputs a tensor of shape `(batch, chunk_size, action_dim)`. Each value is a raw float. By contrast, tokenization-based approaches like RT-1 bin each dimension into hundreds of discrete tokens.


In [3]:
import torch

# Simulate what ACT outputs
batch_size = 1
chunk_size = cfg.chunk_size  # 100
action_dim = 7  # typical: x, y, z, roll, pitch, yaw, gripper

actions = torch.randn(batch_size, chunk_size, action_dim)
print(f"ACT action shape:  {actions.shape}")
print(f"Total values:      {actions.numel()}")  # 700
print(f"Value range:       [{actions.min():.2f}, {actions.max():.2f}]")
print(f"Data type:         {actions.dtype}")

# Compare: if this were RT-1 style binning (256 bins/dim)
tokens_if_binned = chunk_size * action_dim  # 700 tokens
print(f"\nIf binning (256 bins/dim): {tokens_if_binned} tokens per chunk")
print(f"ACT uses 0 tokens: continuous vectors instead")


ACT action shape:  torch.Size([1, 100, 7])
Total values:      700
Value range:       [-2.92, 3.04]
Data type:         torch.float32

If binning (256 bins/dim): 700 tokens per chunk
ACT uses 0 tokens — continuous vectors instead


### 4. CVAE: the stochastic action head

ACT uses stochastic generation through a learned distribution. The CVAE encodes observations into a latent distribution (μ, σ), samples z, and decodes into action chunks. This captures multi-modal action distributions (for example, you could go left or right around an obstacle).


In [4]:
# ACT uses a CVAE (Conditional Variational Autoencoder)
# Encoder: observation -> latent distribution (mu, sigma)
# Sample: z ~ N(mu, sigma)
# Decoder: z -> action chunk (chunk_size, action_dim)

# The loss = reconstruction_loss + kl_divergence
# This allows ACT to model MULTIPLE valid action trajectories
# for the same observation — multimodal action distributions.

print("ACT CVAE Flow:")
print("  Observation -> Encoder -> (μ, σ) -> Sample z")
print("  z -> Decoder -> Action chunk (100 × 7 continuous values)")

# Compare with tokenization-based approaches:
print("\nContrast with tokenization VLAs:")
print("  RT-2: Observation -> LLM -> Token IDs -> Binned action values")
print("  pi0-FAST: Observation -> VLM -> FAST tokens -> Inverse DCT")


ACT CVAE Flow:
  Observation -> Encoder -> (μ, σ) -> Sample z
  z -> Decoder -> Action chunk (100 × 7 continuous values)

Contrast with tokenization VLAs:
  RT-2: Observation -> LLM -> Token IDs -> Binned action values
  pi0-FAST: Observation -> VLM -> FAST tokens -> Inverse DCT


### 5. Temporal ensemble (smoothing)

ACT uses temporal ensembling to smooth consecutive action chunks. Overlapping chunks are averaged with exponential weighting.


In [5]:
# Temporal ensemble: when chunks overlap, average them
# If chunk_1 predicts actions [a0..a99] and chunk_2 predicts [a50..a149],
# actions a50..a99 are averaged with exponential decay weighting.

print("Temporal Ensemble:")
print("  Chunk 1: t=0..99")
print("  Chunk 2:        t=50..149")
print("  Overlap: t=50..99 averaged with exp(-Δt/τ) weights")


Temporal Ensemble:
  Chunk 1: t=0..99
  Chunk 2:        t=50..149
  Overlap: t=50..99 averaged with exp(-Δt/τ) weights


### The Bottom Line

ACT represents actions as **continuous vectors with learned distributions**. The CVAE handles action multimodality by sampling from a learned latent space. This was the dominant paradigm before VLAs entered the picture.
